In [3]:
import torch
import torch.nn as nn

In [5]:
class SqueezeExcite(nn.Module):
    def __init__(self, in_channels, se_ratio=0.25):
        super(SqueezeExcite, self).__init__()
        hidden = max(1, int(in_channels * se_ratio))
        self.se = nn.Sequential(
                        nn.AdaptiveAvgPool2d(1),
                        nn.Conv2d(in_channels, hidden, 1),
                        nn.SiLU(inplace=True),
                        nn.Conv2d(hidden, in_channels, 1),
                        nn.Sigmoid()
        )

    def forward(self,x):
        return x * self.se(x)                                

In [20]:
class MBConv(nn.Module):
    def __init__(self, expansion, in_channels, out_channels, stride, kernel, se_ratio=0.25):
        super(MBConv, self).__init__()
        hidden = in_channels * expansion
        self.use_residual = (stride == 1 and in_channels == out_channels)
        layers = []

        # Expansion
        if expansion != 1:
            layers += [nn.Conv2d(in_channels, hidden, 1, bias=False), nn.BatchNorm2d(hidden), nn.SiLU(inplace=True)]
        else:
            hidden = in_channels

        # Depthwise
        layers += [nn.Conv2d(hidden, hidden, kernel, stride, padding=kernel//2, groups=hidden, bias=False)]
        # SE
        layers.append(SqueezeExcite(hidden, se_ratio))
        # Projection (linear bottleneck)
        layers += [nn.Conv2d(hidden, out_channels, 1, bias=False), nn.BatchNorm(out_channels)]
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        out = self.block(x)
        if(self.use_residual):
            out = out + x

        return out

In [23]:
@staticmethod
def adjust_channels(channel, width_mult, divisor=8):
    channel *= width_mult
    return int(math.ceil(channel / divisor) * devisor)
@staticmethod
def adjust_depth(r, depth_mult):
    return int(math.ceil(r*depth_mult))

In [ ]:
class EfficientNet(nn.Module):
    B0_CONFIG = [
        # expansion, out_channels, repeats, stride, kernel
        (1,  16, 1, 1, 3),
        (6,  24, 2, 2, 3),
        (6,  40, 2, 2, 5),
        (6,  80, 3, 2, 3),
        (6, 112, 3, 1, 5),
        (6, 192, 4, 2, 5),
        (6, 320, 1, 1, 3),
    ]

    def __init__(self, phi=0, num_classes=1000):
        super(EfficientNet, self).__init__()

        # Scaling coefficients
        beta = 1.1 # width
        alpha = 1.2 # depth

        depth_mult = alpha ** phi
        width_mult = beta ** phi

        # Stem
        in_channels = adjust_width(32, width_mult)
        self.stem = nn.Sequential(
                        nn.Conv2d(3, in_channels, 3, stride=2, padding=1, bias=False),
                        nn.BatchNorm2d(in_channels),
                        nn.SiLU(inplace=True)
        )

        blocks = []
        for expand, out_channels, depth, stride, k in self.B0_CONFIG:
            out_channels = adjust_width(out_channels, width_mult)
            depth = adjust_depth(repeats, depth_mult)

            for i in range(depth):
                blocks.append(MBConv(expanion=expand, in_channels=in_channels, out_channels=out_channels,
                                     stride=stride if i == 0 else =1,
                                     kernel=k
                                    )
                             )
                in_channels = out_channels
        self.blocks = nn.Sequential(*block)

        head_channel = adjust_width(1280, width_mult)
        self.head = nn.Sequential(
                    nn.Conv2d(in_channel, head_channel, 1, bias=False),
                    nn.BatchNorm(head_channels),
                    nn.SiLU(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(head_channel, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.pool(x).flatten(1)
        out = self.classifier(x)
        return out
        